In [1]:
import getpass
import os
deepseek_api_key = "sk-f46c7e2053764299ace45b6e3d98de77"

if not os.getenv("DEEPSEEK_API_KEY"):
    os.environ["DEEPSEEK_API_KEY"] = deepseek_api_key

In [70]:
base_url = "https://api.deepseek.com"


In [3]:


from langchain_deepseek import ChatDeepSeek

llm = ChatDeepSeek(
    model="deepseek-chat",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
    # other params...
)

messages = [
    (
        "system",
        "You are a helpful assistant that translates English to French. Translate the user sentence.",
    ),
    ("human", "I love programming."),
]
ai_msg = llm.invoke(messages)
ai_msg.content

"J'adore programmer."

# Langchain tools & agent executor
- 构建一个agent,可以自己调用tools进行数学计算

In [4]:
import numpy as np
from langchain_core.tools import Tool
from langchain_core.tools import tool

In [5]:
@tool
def add(num1: float, num2: float) -> float:
    "Add two numbers."
    return num1 + num2
    
@tool
def subtract(num1: float, num2: float) -> float:
    """
    Subtract two numbers.
    """
    return num1 - num2
    
@tool
def multiply(num1: float, num2: float) -> float:
    """Multiply two float ."""
    return num1 * num2

@tool
def divide(numerator: float, denominator: float) -> float:
    """
    Divides the numerator by the denominator.
    """

    result = numerator / denominator
    return result

@tool
def power(base: float, exponent: float) -> float:
    "Take the base to the exponent power, base^exponent."
    return base**exponent


@tool
def exp(x):
    """
    Calculate the natural exponential $e^x$
    """
    return np.exp(x)

In [6]:
tools = [add, subtract, multiply, divide, power, exp]

In [7]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_openai_tools_agent
from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder,
    HumanMessagePromptTemplate,
    SystemMessagePromptTemplate,
)

In [8]:
# llm = ChatOpenAI(model="gpt-4o", temperature=0, streaming=True)

system_template = """
You are a helpful math assistant that uses calculation functions to solve complex math problems step by step.
"""

human_template = "{input}"

prompt = ChatPromptTemplate.from_messages(
    [
        SystemMessagePromptTemplate.from_template(system_template),
        MessagesPlaceholder(variable_name="chat_history", optional=True),
        HumanMessagePromptTemplate.from_template(input_variables=["input"], template=human_template),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ]
)

agent = create_openai_tools_agent(llm, tools, prompt)

In [9]:
from langchain.agents import AgentExecutor

agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [10]:
agent_executor.invoke({"input": "What is the result of directive of sigmoid(2.5)?"})



> Entering new AgentExecutor chain...

Invoking: `exp` with `{'x': -2.5}`
responded: The sigmoid function is defined as:

\[
\sigma(x) = \frac{1}{1 + e^{-x}}
\]

To find the derivative of the sigmoid function, we can use the following property:

\[
\sigma'(x) = \sigma(x) \cdot (1 - \sigma(x))
\]

Let's compute the derivative at \( x = 2.5 \):

1. First, compute \( \sigma(2.5) \).
2. Then, use the formula above to find \( \sigma'(2.5) \).

I'll calculate this step by step.

0.0820849986238988
Invoking: `add` with `{'num1': 1, 'num2': 0.0820849986238988}`


1.0820849986238987
Invoking: `divide` with `{'numerator': 1, 'denominator': 1.0820849986238987}`


0.9241418199787566
Invoking: `subtract` with `{'num1': 1, 'num2': 0.9241418199787566}`


0.07585818002124345
Invoking: `multiply` with `{'num1': 0.9241418199787566, 'num2': 0.07585818002124345}`


0.07010371654510807The derivative of the sigmoid function at \( x = 2.5 \) is approximately:

\[
\sigma'(2.5) \approx 0.0701
\]

> Finished 

{'input': 'What is the result of directive of sigmoid(2.5)?',
 'output': "The derivative of the sigmoid function at \\( x = 2.5 \\) is approximately:\n\n\\[\n\\sigma'(2.5) \\approx 0.0701\n\\]"}

In [12]:
import torch
import torch.nn.functional as F
F.sigmoid(torch.tensor([2.5])) * (1-F.sigmoid(torch.tensor([2.5])))

tensor([0.0701])

# conversational agent

In [13]:
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain.agents import AgentExecutor, create_react_agent

In [19]:
@tool
def get_employee_id(name):
  """
  To get employee id, it takes employee name as arguments
  name(str): Name of the employee
  """
  fake_employees = {
    "Alice": "E001", "Bob": "E002", "Charlie": "E003",
    "Diana": "E004", "yoli": "E005", "Fiona": "E006",
    "George": "E007", "Hannah": "E008", "Ian": "E009",
    "Jasmine": "E010"
  }
  
  return fake_employees.get(name, "Employee not found")

# Custom tool for the Agent 
# ValueError: Function must have a docstring if description not provided.
@tool
def get_employee_salary(employee_id):
  """
  To get the salary of an employee, it takes employee_id as input and return salary
  """
  employee_salaries = {
    "E001": 56000, "E002": 47000, "E003": 52000,
    "E004": 61000, "E005": 45000, "E006": 58000,
    "E007": 49000, "E008": 53000, "E009": 50000,
    "E010": 55000
    }
  return employee_salaries.get(employee_id,"Employee not found")

In [20]:
from langchain import hub
prompt = hub.pull("hwchase17/react")

/home/yundai/tfenv/lib/python3.10/site-packages/langsmith/client.py:272: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


In [21]:
prompt = hub.pull("hwchase17/react")
# llm = ChatOpenAI(model='gpt-4o')
tools = [get_employee_salary, get_employee_id]
agent = create_react_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)
agent_executor.invoke({"input": "What is the Salary of yoli?"})

/home/yundai/tfenv/lib/python3.10/site-packages/langsmith/client.py:272: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(




> Entering new AgentExecutor chain...
Thought: To find Yoli's salary, I first need to get her employee ID using her name, and then use that ID to retrieve her salary.

Action: get_employee_id  
Action Input: {"name": "yoli"}  
Employee not foundIt seems that there is no employee named "yoli" in the system. Therefore, I cannot retrieve the salary.

Final Answer: No employee named "yoli" found in the system. Salary information is unavailable.

> Finished chain.


{'input': 'What is the Salary of yoli?',
 'output': 'No employee named "yoli" found in the system. Salary information is unavailable.'}

## 显式操作 
- https://python.langchain.com/docs/how_to/migrate_agent/ 

In [22]:
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

# model = ChatOpenAI(model="gpt-4o")


@tool
def magic_function(input: int) -> int:
    """Applies a magic function to an input."""
    return input + 2


tools = [magic_function]


query = "what is the value of magic_function(3)?"

In [23]:
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant"),
        ("human", "{input}"),
        # Placeholders fill up a **list** of messages
        ("placeholder", "{agent_scratchpad}"),
    ]
)


agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools)

agent_executor.invoke({"input": query})

{'input': 'what is the value of magic_function(3)?',
 'output': 'The value of `magic_function(3)` is `5`.'}

## example 2

In [24]:
@tool
def get_word_length(word: str) -> int:
    """Returns the length of a word."""
    return len(word)


get_word_length.invoke("abc")

3

In [25]:
tools = [get_word_length]

In [26]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are very powerful assistant, but don't know current events",
        ),
        ("user", "{input}"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ]
)

In [27]:
llm_with_tools = llm.bind_tools(tools)

In [28]:
from langchain.agents.format_scratchpad.openai_tools import (
    format_to_openai_tool_messages,
)
from langchain.agents.output_parsers.openai_tools import OpenAIToolsAgentOutputParser

# runnable Sequence，根据需要循环执行这 4 步；
agent = (
    {
        "input": lambda x: x["input"],
        "agent_scratchpad": lambda x: format_to_openai_tool_messages(
            x["intermediate_steps"]
        ),
    }
    | prompt
    | llm_with_tools
    | OpenAIToolsAgentOutputParser()
)

In [ ]:
#  )
# result = agent_executor.invoke({"input": query})
# print(result["intermediate_steps"])




In [32]:
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True,return_intermediate_steps=True)

In [34]:
result = agent_executor.invoke({"input": "How many letters in the word eudca"})
print(result["intermediate_steps"])






> Entering new AgentExecutor chain...

Invoking: `get_word_length` with `{'word': 'eudca'}`


5The word "eudca" has 5 letters.

> Finished chain.
[(ToolAgentAction(tool='get_word_length', tool_input={'word': 'eudca'}, log="\nInvoking: `get_word_length` with `{'word': 'eudca'}`\n\n\n", message_log=[AIMessageChunk(content='', additional_kwargs={'tool_calls': [{'index': 0, 'id': 'call_0_cbb11492-790c-4657-b6b6-42a855ee58b6', 'function': {'arguments': '{"word":"eudca"}', 'name': 'get_word_length'}, 'type': 'function'}]}, response_metadata={'finish_reason': 'tool_calls', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_8802369eaa_prod0623_fp8_kvcache'}, id='run--9c76f79a-c390-4de0-b8aa-03baa01f84ea', tool_calls=[{'name': 'get_word_length', 'args': {'word': 'eudca'}, 'id': 'call_0_cbb11492-790c-4657-b6b6-42a855ee58b6', 'type': 'tool_call'}], usage_metadata={'input_tokens': 118, 'output_tokens': 21, 'total_tokens': 139, 'input_token_details': {'cache_read': 64}, 'output_token_detail

In [33]:
process_list = list(agent_executor.stream({"input": "How many letters in the word eudca"}))



> Entering new None chain...

Invoking: `get_word_length` with `{'word': 'eudca'}`


5The word "eudca" has 5 letters.

> Finished chain.


In [35]:
process_list

[{'actions': [ToolAgentAction(tool='get_word_length', tool_input={'word': 'eudca'}, log="\nInvoking: `get_word_length` with `{'word': 'eudca'}`\n\n\n", message_log=[AIMessageChunk(content='', additional_kwargs={'tool_calls': [{'index': 0, 'id': 'call_0_0fcab5fb-960f-4325-95d5-3dfda938b1b6', 'function': {'arguments': '{"word":"eudca"}', 'name': 'get_word_length'}, 'type': 'function'}]}, response_metadata={'finish_reason': 'tool_calls', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_8802369eaa_prod0623_fp8_kvcache'}, id='run--bb1c84f3-953a-4ba4-8dd5-e5a22d96da73', tool_calls=[{'name': 'get_word_length', 'args': {'word': 'eudca'}, 'id': 'call_0_0fcab5fb-960f-4325-95d5-3dfda938b1b6', 'type': 'tool_call'}], usage_metadata={'input_tokens': 118, 'output_tokens': 21, 'total_tokens': 139, 'input_token_details': {'cache_read': 64}, 'output_token_details': {}}, tool_call_chunks=[{'name': 'get_word_length', 'args': '{"word":"eudca"}', 'id': 'call_0_0fcab5fb-960f-4325-95d5-3dfda938b1b6', 

In [37]:
from langchain_core.prompts import MessagesPlaceholder

MEMORY_KEY = "chat_history"
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are very powerful assistant, but bad at calculating lengths of words.",
        ),
        MessagesPlaceholder(variable_name=MEMORY_KEY),
        ("user", "{input}"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ]
)

In [38]:
from langchain_core.messages import AIMessage, HumanMessage

chat_history = []

In [39]:
agent = (
    {
        "input": lambda x: x["input"],
        "agent_scratchpad": lambda x: format_to_openai_tool_messages(
            x["intermediate_steps"]
        ),
        "chat_history": lambda x: x["chat_history"],
    }
    | prompt
    | llm_with_tools
    | OpenAIToolsAgentOutputParser()
)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [42]:
input1 = "how many letters in the word bloom?"
result = agent_executor.invoke({"input": input1, 
                                "chat_history": chat_history})
chat_history.extend(
    [
        HumanMessage(content=input1),
        AIMessage(content=result["output"]),
    ]
)
input_2 = "is that a real word?"

result = agent_executor.invoke({"input": input_2, 
                       "chat_history": chat_history})
chat_history.extend(
    [
        HumanMessage(content=input_2),
        AIMessage(content=result["output"]),
    ]
)



> Entering new AgentExecutor chain...

Invoking: `get_word_length` with `{'word': 'bloom'}`


5The word "bloom" has 5 letters.

> Finished chain.


> Entering new AgentExecutor chain...
Yes, "bloom" is a real word. It can refer to:

1. The flower of a plant (noun).
2. The process of flowering (verb).
3. A state or time of beauty, freshness, or vigor (e.g., "the bloom of youth").

Would you like me to confirm the length of another word?

> Finished chain.


In [43]:
chat_history

[HumanMessage(content='how many letters in the word educa?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='The word "educa" has 5 letters.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='how many letters in the word bloom?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='The word "bloom" has 5 letters.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='is that a real word?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Yes, "bloom" is a real word. It can refer to:\n\n1. The flower of a plant (noun).\n2. The process of flowering (verb).\n3. A state or time of beauty, freshness, or vigor (e.g., "the bloom of youth").\n\nWould you like me to confirm the length of another word?', additional_kwargs={}, response_metadata={})]

# Langraph basics学习

In [44]:
from langchain.agents import create_openai_functions_agent
from langchain_core.utils.function_calling import convert_to_openai_function
from langchain_core.agents import AgentAction, AgentFinish
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage, FunctionMessage
from langgraph.graph import END, StateGraph, MessagesState
from langchain.tools.render import format_tool_to_openai_function
from langchain.tools import BaseTool, StructuredTool, Tool, tool
import os
from typing import TypedDict, Annotated, List, Union, Sequence
import operator
import json

In [57]:
from langgraph.prebuilt import ToolNode

In [45]:
class AgentState(TypedDict):
   input: str
   chat_history: list[BaseMessage]
   agent_outcome: Union[AgentAction, AgentFinish, None]
    # 不断追加和传递
   intermediate_steps: Annotated[list[tuple[AgentAction, str]], operator.add]

In [46]:
import random

@tool("upper_case", return_direct=True)
def to_upper_case(input:str) -> str:
  """Returns the input as all upper case."""
  return input.upper()

@tool("random_number", return_direct=True)
def random_number_maker(input:str) -> str:
    """Returns a random number between 0-100."""
    return random.randint(0, 100)

tools = [to_upper_case,random_number_maker]

In [48]:
random_number_maker.run("random_number")

98

In [49]:
to_upper_case.run("Piitty")

'PIITTY'

## Agent

In [50]:
prompt = hub.pull('hwchase17/openai-functions-agent')
prompt

/home/yundai/tfenv/lib/python3.10/site-packages/langsmith/client.py:272: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='

In [51]:
print(prompt)

input_variables=['agent_scratchpad', 'input'] optional_variables=['chat_history'] input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')]

In [ ]:
agent = create_openai_functions_agent(llm, tools, prompt)

In [52]:
type(agent)

langchain_core.runnables.base.RunnableSequence

In [53]:
inputs = {"input": "give me a random number and then write in words and make it upper case.",
          "chat_history": [],
          "intermediate_steps":[]}

agent_outcome = agent.invoke(inputs)

In [55]:
agent_outcome.__dict__

{'return_values': {'output': "Here's a random number: **7**.  \n\nIn words and uppercase: **SEVEN**."},
 'log': "Here's a random number: **7**.  \n\nIn words and uppercase: **SEVEN**.",
 'type': 'AgentFinish'}

## tool

In [58]:
# tool_executor = ToolExecutor(tools)
tool_executor = ToolNode(tools)

In [59]:
def run_agent(state):
    agent_outcome = agent.invoke(state)
    # AgentState.agent_outcome: 当前输出，决策动作
    return {"agent_outcome": agent_outcome}

def execute_tools(state):
    agent_action = state['agent_outcome']
    output = tool_executor.invoke(agent_action)
    print(f"The agent action is {agent_action}")
    print(f"The tool result is: {output}")
    # AgentState.intermediate_steps: 追加逻辑
    return {"intermediate_steps": [(agent_action, str(output))]}

def should_continue(state):
    if isinstance(state['agent_outcome'], AgentFinish):
        return "end"
    return "continue"

## graph

In [60]:
workflow = StateGraph(AgentState)

workflow.add_node("agent", run_agent)
workflow.add_node("action", execute_tools)

workflow.set_entry_point("agent")

# 条件分支
workflow.add_conditional_edges(
    "agent",
    should_continue,
    {
        "continue": "action",
        "end": END
    }
)

workflow.add_edge('action', 'agent')

graph = workflow.compile()

In [61]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    # This requires some extra dependencies and is optional
    pass

In [62]:
inputs = {"input": "give me a random number and then write in words and make it upper case.", 
          "chat_history": []}
for s in graph.stream(inputs):
    print(list(s.values())[0])
    print("----")

{'agent_outcome': AgentFinish(return_values={'output': "Here's a random number: **7**.  \n\nIn words and uppercase: **SEVEN**."}, log="Here's a random number: **7**.  \n\nIn words and uppercase: **SEVEN**.")}
----


In [63]:
inputs = {"input": "give me a random number and then write in words and make it upper case", "chat_history": []}

output = graph.invoke(inputs)

In [64]:
output

{'input': 'give me a random number and then write in words and make it upper case',
 'chat_history': [],
 'agent_outcome': AgentFinish(return_values={'output': "Here's a random number: **7**.  \n\nIn words and uppercase: **SEVEN**."}, log="Here's a random number: **7**.  \n\nIn words and uppercase: **SEVEN**."),
 'intermediate_steps': []}

In [65]:
output.get('agent_outcome')

AgentFinish(return_values={'output': "Here's a random number: **7**.  \n\nIn words and uppercase: **SEVEN**."}, log="Here's a random number: **7**.  \n\nIn words and uppercase: **SEVEN**.")

In [66]:
output['intermediate_steps']

[]

# Tool calling agents

In [68]:
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools import WikipediaQueryRun

api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=100)
tool = WikipediaQueryRun(api_wrapper=api_wrapper)
{'name': tool.name,
 'description': tool.description,
 'input': tool.args,
 'return_direct': tool.return_direct,
 }

{'name': 'wikipedia',
 'description': 'A wrapper around Wikipedia. Useful for when you need to answer general questions about people, places, companies, facts, historical events, or other subjects. Input should be a search query.',
 'input': {'query': {'description': 'query to look up on wikipedia',
   'title': 'Query',
   'type': 'string'}},
 'return_direct': False}

In [ ]:
tool.run({"query": "langchain"})

In [ ]:
from openai import OpenAI
from datetime import datetime
import json
import os

client = OpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"), # 如果您没有配置环境变量，请在此处用您的API Key进行替换
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",  # 填写DashScope SDK的base_url
)

In [ ]:
# 定义工具列表，模型在选择使用哪个工具时会参考工具的name和description
tools = [
    # 工具1 获取当前时刻的时间
    {
        "type": "function",
        "function": {
            "name": "get_current_time",
            "description": "当你想知道现在的时间时非常有用。",
            "parameters": {}  # 因为获取当前时间无需输入参数，因此parameters为空字典
        }
    },  
    # 工具2 获取指定城市的天气
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": "当你想查询指定城市的天气时非常有用。",
            "parameters": {  # 查询天气时需要提供位置，因此参数设置为location
                        "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "城市或县区，比如北京市、杭州市、余杭区等。"
                    }
                }
            },
            "required": [
                "location"
            ]
        }
    }
]

# 模拟天气查询工具。返回结果示例：“北京今天是晴天。”
def get_current_weather(location):
    return f"{location}今天是雨天。 "

# 查询当前时间的工具。返回结果示例：“当前时间：2024-04-15 17:15:18。“
def get_current_time():
    # 获取当前日期和时间
    current_datetime = datetime.now()
    # 格式化当前日期和时间
    formatted_time = current_datetime.strftime('%Y-%m-%d %H:%M:%S')
    # 返回格式化后的当前时间
    return f"当前时间：{formatted_time}。"